In [2]:
import os, yaml, sys
import torch
from einops import reduce, rearrange
import matplotlib.pyplot as plt
import numpy as np
ENV = os.getenv("MY_ENV", "dev")
with open("../../config.yaml", "r") as f:
    config = yaml.safe_load(f)
paths = config[ENV]["paths"]
sys.path.append(paths["src_path"])
sys.path.append(paths["useful_stuff_path"])
# from useful_stuff.image_processing.utils import 
from useful_stuff.image_processing.computational_models import imgANN, load_model, get_relevant_output_layers, get_activation
from useful_stuff.general_utils.utils import get_device, get_module_by_path
device = get_device()

In [3]:
from dataclasses import dataclass, field

@dataclass
class Cfg:
    model_name = "dino_v3_l";
    pkg = 'hf'
    input_size = 384
    pooling = 'all'
    model_url = "facebook/dinov3-vitl16-pretrain-lvd1689m"
cfg = Cfg()

In [4]:
m = imgANN(cfg.model_name, cfg.pkg,  cfg.input_size, dtype=torch.float32, attn_implementation='sdpa', repo_url=cfg.model_url)
m.create_forward_hook()

17:14:25 - device being used: mps


({},
 {'layer.0.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b6cf0e0>,
  'layer.1.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31a5d1090>,
  'layer.2.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x319f08190>,
  'layer.3.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b7b02b0>,
  'layer.4.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b7b0180>,
  'layer.5.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b749370>,
  'layer.6.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b71acf0>,
  'layer.7.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31a651480>,
  'layer.8.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31fd54850>,
  'layer.9.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31fd54750>,
  'layer.10.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b6bad50>,
  'layer.11.mlp.down_proj': <torch.utils.hooks.RemovableHandle at 0x31b6ba5d0>,
  'layer.12.mlp.down_proj': <torch.utils.hook

In [16]:
from torchvision import transforms
from PIL import Image

# Load a sample image and preprocess it for AlexNet
import urllib.request

# Download a sample image
url = "https://www.python.org/static/community_logos/python-logo.png"
urllib.request.urlretrieve(url, "sample_image.jpg")
image = Image.open("sample_image.jpg")

# Define preprocessing for AlexNet (ImageNet normalization)
preprocess = transforms.Compose([
    transforms.Resize((cfg.input_size, cfg.input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Preprocess and add batch dimension
input_tensor = preprocess(image).unsqueeze(0).to(device)

# Forward pass
with torch.no_grad():
    output = m.model(input_tensor)



# Check captured features
for layer_name, feature in m.features.items():
    print(f"{layer_name}: {feature.shape}")

layer.0.mlp.down_proj: torch.Size([1, 594944])
layer.1.mlp.down_proj: torch.Size([1, 594944])
layer.2.mlp.down_proj: torch.Size([1, 594944])
layer.3.mlp.down_proj: torch.Size([1, 594944])
layer.4.mlp.down_proj: torch.Size([1, 594944])
layer.5.mlp.down_proj: torch.Size([1, 594944])
layer.6.mlp.down_proj: torch.Size([1, 594944])
layer.7.mlp.down_proj: torch.Size([1, 594944])
layer.8.mlp.down_proj: torch.Size([1, 594944])
layer.9.mlp.down_proj: torch.Size([1, 594944])
layer.10.mlp.down_proj: torch.Size([1, 594944])
layer.11.mlp.down_proj: torch.Size([1, 594944])
layer.12.mlp.down_proj: torch.Size([1, 594944])
layer.13.mlp.down_proj: torch.Size([1, 594944])
layer.14.mlp.down_proj: torch.Size([1, 594944])
layer.15.mlp.down_proj: torch.Size([1, 594944])
layer.16.mlp.down_proj: torch.Size([1, 594944])
layer.17.mlp.down_proj: torch.Size([1, 594944])
layer.18.mlp.down_proj: torch.Size([1, 594944])
layer.19.mlp.down_proj: torch.Size([1, 594944])
layer.20.mlp.down_proj: torch.Size([1, 594944])
la

In [17]:
np.all(np.isnan(m.features['layer.0.mlp.down_proj'].cpu().numpy()))

np.False_

In [18]:
# m.extract_features(input_tensor, 'fx')
m.extract_features({"pixel_values": input_tensor}, 'hook')

{'layer.0.mlp.down_proj': tensor([[ 0.1669, -0.1894,  0.0714,  ...,  0.1561,  0.3949,  0.2974]],
        device='mps:0'),
 'layer.1.mlp.down_proj': tensor([[-0.0892, -0.2544, -0.2863,  ...,  0.0023,  0.3708, -0.2370]],
        device='mps:0'),
 'layer.2.mlp.down_proj': tensor([[ 1.6360, -0.1276, -0.2136,  ..., -0.2487, -0.1820, -0.1170]],
        device='mps:0'),
 'layer.3.mlp.down_proj': tensor([[ 0.0123,  0.6931, -0.2591,  ...,  0.1461,  0.1938, -0.1808]],
        device='mps:0'),
 'layer.4.mlp.down_proj': tensor([[-2.8594,  0.2214,  0.0284,  ...,  0.1527,  0.2227,  0.9867]],
        device='mps:0'),
 'layer.5.mlp.down_proj': tensor([[ 0.3159, -0.7101,  0.4601,  ...,  0.3273,  0.8558, -0.7711]],
        device='mps:0'),
 'layer.6.mlp.down_proj': tensor([[-3.6887,  0.3204,  0.1809,  ..., -0.5268, -0.0352, -0.9461]],
        device='mps:0'),
 'layer.7.mlp.down_proj': tensor([[ 0.9709, -0.4414,  0.0221,  ..., -1.4238, -0.5149, -1.7967]],
        device='mps:0'),
 'layer.8.mlp.down_proj'